## ایمبیڈنگز

ہمارے پچھلے مثال میں، ہم نے لمبائی `vocab_size` والے اعلیٰ جہتی بیگ آف ورڈز ویکٹرز پر کام کیا، اور ہم واضح طور پر کم جہتی پوزیشنل نمائندگی کے ویکٹرز کو اسپارس ون-ہاٹ نمائندگی میں تبدیل کر رہے تھے۔ یہ ون-ہاٹ نمائندگی میموری مؤثر نہیں ہے، اس کے علاوہ، ہر لفظ کو ایک دوسرے سے آزاد سمجھا جاتا ہے، یعنی ون-ہاٹ انکوڈڈ ویکٹر الفاظ کے درمیان کوئی سماجی مماثلت ظاہر نہیں کرتے۔

اس یونٹ میں، ہم **News AG** ڈیٹاسیٹ کو مزید دریافت کریں گے۔ شروع کرنے کے لیے، آئیے ڈیٹا لوڈ کریں اور پچھلے نوٹ بک سے کچھ تعریفیں حاصل کریں۔


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## ایمبیڈنگ کیا ہے؟

**ایمبیڈنگ** کا خیال یہ ہے کہ الفاظ کو کم جہتی کثیف ویکٹروں کے ذریعہ ظاہر کیا جائے، جو کسی طرح ایک لفظ کے معنی کو ظاہر کرتے ہیں۔ ہم بعد میں بات کریں گے کہ کس طرح معنی خیز وورد ایمبیڈنگز بنانے ہیں، لیکن فی الحال ایمبیڈنگز کو بس ایک طریقہ سمجھیں کہ کس طرح ایک لفظ کے ویکٹر کی جہت کو کم کیا جائے۔

تو، ایمبیڈنگ لئیر ایک لفظ کو ان پٹ کے طور پر لے گا، اور مخصوص کردہ `embedding_size` کا ایک آؤٹ پٹ ویکٹر پیدا کرے گا۔ ایک لحاظ سے، یہ `Linear` لئیر کی طرح ہے، لیکن ایک ہاٹ انکوڈڈ ویکٹر لینے کی بجائے، یہ ایک لفظ نمبر کو ان پٹ کے طور پر لے سکے گا۔

ہمارے نیٹ ورک میں ایمبیڈنگ لئیر کو پہلی لئیر کے طور پر استعمال کر کے، ہم بیگ آف ورڈز سے **ایمبیڈنگ بیگ** ماڈل پر سوئچ کر سکتے ہیں، جہاں ہم سب سے پہلے اپنے متن میں ہر لفظ کو اس کے مطابق ایمبیڈنگ میں تبدیل کرتے ہیں، اور پھر ان سب ایمبیڈنگز پر کوئی مجموعی فَنکشن حساب کرتے ہیں، جیسے کہ `sum`، `average` یا `max`۔

![پانچ تسلسل کلمات کے لیے ایک ایمبیڈنگ کلاسیفائر دکھاتا ہوا تصویر۔](../../../../../translated_images/ur/embedding-classifier-example.b77f021a7ee67eee.webp)

ہمارا کلاسیفائر نیورل نیٹ ورک ایمبیڈنگ لئیر سے شروع ہوگا، پھر ایگریگیشن لئیر، اور اس کے اوپر ایک لینیئر کلاسیفائر لگے گا:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### متغیر تسلسل کے سائز سے نمٹنا

اس فن تعمیر کے نتیجے کے طور پر، ہمارے نیٹ ورک کے لیے منی بیچز ایک خاص طریقے سے بنائے جانے چاہئیں۔ پچھلے یونٹ میں، جب بیگ آف ورڈز استعمال کر رہے تھے، تو منی بیچ میں تمام BoW ٹینسرز کا سائز برابر تھا `vocab_size`، چاہے ہمارے متن کی تسلسل کی اصل لمبائی کچھ بھی ہو۔ جب ہم ورڈ ایمبیڈنگز کی طرف بڑھیں گے، تو ہر متن کے نمونے میں الفاظ کی تعداد متغیر ہوگی، اور جب ان نمونوں کو منی بیچز میں یکجا کریں گے تو ہمیں کچھ پیڈنگ لگانی ہوگی۔

یہ کام ڈاٹاسیٹس کو `collate_fn` فنکشن فراہم کرنے کی اسی تکنیک استعمال کرکے کیا جا سکتا ہے:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### ایمبیڈنگ کلاسیفائر کی تربیت

اب جب کہ ہم نے مناسب ڈیٹالاڈر متعین کر لیا ہے، ہم اس ماڈل کو پچھلے یونٹ میں متعین کیے گئے تربیتی فنکشن کا استعمال کرتے ہوئے تربیت دے سکتے ہیں:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **نوٹ**: ہم یہاں صرف 25 ہزار ریکارڈز کے لیے تربیت دے رہے ہیں (ایک مکمل ایپوک سے کم) وقت کی بچت کے لیے، لیکن آپ تربیت جاری رکھ سکتے ہیں، کئی ایپوک کے لیے تربیت دینے کے لیے ایک فنکشن لکھ سکتے ہیں، اور زیادہ درستگی حاصل کرنے کے لیے لرننگ ریٹ پیرامیٹر کے ساتھ تجربہ کر سکتے ہیں۔ آپ تقریباً 90٪ کی درستگی تک پہنچنے کے قابل ہونا چاہیے۔


### ایمبیڈنگ بیگ لیئر اور متغیر لمبائی سیکوینس کی نمائندگی

پچھلے آرکیٹیکچر میں، ہمیں منی بیچ میں فٹ کرنے کے لیے تمام سیکوینسز کو ایک ہی لمبائی تک پیڈ کرنا پڑتا تھا۔ یہ متغیر لمبائی والے سیکوینسز کی نمائندگی کرنے کا سب سے مؤثر طریقہ نہیں ہے - ایک اور طریقہ **آفسیٹ** ویکٹر کا استعمال کرنا ہوگا، جو ایک بڑے ویکٹر میں محفوظ کیے گئے تمام سیکوینسز کے آفسیٹس کو رکھے گا۔

![آفسیٹ سیکوینس کی نمائندگی دکھاتی ہوئی تصویر](../../../../../translated_images/ur/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **نوٹ**: اوپر دی گئی تصویر میں ہم کرداروں کی ایک سیکوینس دکھاتے ہیں، لیکن ہمارے مثال میں ہم الفاظ کی سیکوینسز کے ساتھ کام کر رہے ہیں۔ تاہم، سیکوینسز کو آفسیٹ ویکٹر کے ساتھ پیش کرنے کا عمومی اصول ایک ہی رہتا ہے۔

آفسیٹ نمائندگی کے ساتھ کام کرنے کے لیے، ہم [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) لیئر استعمال کرتے ہیں۔ یہ `Embedding` کے مشابہ ہے، لیکن یہ مواد کے ویکٹر اور آفسیٹ ویکٹر کو ان پٹ کے طور پر لیتا ہے، اور اس میں اوسط نکالنے والی لیئر بھی شامل ہوتی ہے، جو `mean`، `sum` یا `max` ہو سکتی ہے۔

یہاں ترمیم شدہ نیٹ ورک ہے جو `EmbeddingBag` استعمال کرتا ہے:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

ڈیٹاسیٹ کی تیاری کے لیے، ہمیں ایک تبدیلی فنکشن فراہم کرنا ہوگا جو آفسیٹ ویکٹر کی تیاری کرے گا:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

نوٹ کریں کہ، پچھلے تمام مثالوں کے برخلاف، ہمارا نیٹ ورک اب دو پیرامیٹرز قبول کرتا ہے: ڈیٹا ویکٹر اور آفسیٹ ویکٹر، جو مختلف سائز کے ہوتے ہیں۔ اسی طرح، ہمارا ڈیٹا لوڈر بھی ہمیں 2 کی بجائے 3 ویلیوز فراہم کرتا ہے: دونوں ٹیکسٹ اور آفسیٹ ویکٹرز کو فیچرز کے طور پر مہیا کیا جاتا ہے۔ لہٰذا، ہمیں اپنے تربیتی فنکشن میں تھوڑی سی تبدیلی کرنی ہوگی تاکہ اس کا خیال رکھا جا سکے:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## معنوی انبیڈنگ: ورڈ2 ویک

ہمارے پچھلے مثال میں، ماڈل انبیڈنگ پرت نے الفاظ کو ویکٹر نمائندگی میں تبدیل کرنا سیکھا، تاہم، اس نمائندگی کا کوئی خاص معنوی مطلب نہیں تھا۔ یہ اچھا ہوتا کہ ایسی ویکٹر نمائندگی سیکھی جائے، جہاں ملتے جلتے الفاظ یا مترادفات ایسے ویکٹروں کے برابر ہوں جو ویکٹر فاصلہ (مثلاً یوکلیڈین فاصلہ) کے لحاظ سے ایک دوسرے کے قریب ہوں۔

ایسا کرنے کے لیے، ہمیں اپنے انبیڈنگ ماڈل کو ایک بڑی متن کی مجموعہ پر مخصوص طریقے سے پری ٹرین کرنا ہوگا۔ معنوی انبیڈنگ سیکھنے کے پہلے طریقوں میں سے ایک [Word2Vec](https://en.wikipedia.org/wiki/Word2vec) کہلاتا ہے۔ یہ دو بنیادی ساختوں پر مبنی ہے جو الفاظ کی تقسیم شدہ نمائندگی پیدا کرنے کے لیے استعمال ہوتے ہیں:

 - **مسلسل بَیگ آف ورڈز** (CBoW) — اس ساخت میں، ہم ماڈل کو گرد وپیش کے سیاق و سباق سے ایک لفظ کی پیشن گوئی کرنے کے لیے تربیت دیتے ہیں۔ اگر ن-گرام $(W_{-2},W_{-1},W_0,W_1,W_2)$ دیا جائے، تو ماڈل کا ہدف $W_0$ کو $(W_{-2},W_{-1},W_1,W_2)$ سے پیش گوئی کرنا ہوتا ہے۔
 - **مسلسل سکپ گرام** CBoW کے برعکس ہے۔ ماڈل موجودہ لفظ کی پیشن گوئی کے لیے گرد وپیش کی کانٹیکسٹ الفاظ کی ونڈو استعمال کرتا ہے۔

CBoW تیز تر ہے، جبکہ سکپ گرام سست ہے، لیکن کم عام الفاظ کی بہتر نمائندگی کرتا ہے۔

![تصویر جو CBoW اور سکپ گرام الگورتھمز کو الفاظ کو ویکٹروں میں تبدیل کرتے ہوئے دکھاتی ہے۔](../../../../../translated_images/ur/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

گوگل نیوز ڈیٹاسیٹ پر پری ٹرین شدہ word2vec انبیڈنگ کے ساتھ تجربہ کرنے کے لیے، ہم **gensim** لائبریری استعمال کر سکتے ہیں۔ نیچے 'neural' سے سب سے زیادہ مشابہ الفاظ تلاش کیے گئے ہیں۔

> **نوٹ:** جب آپ پہلی بار ورڈ ویکٹر بناتے ہیں تو انہیں ڈاؤن لوڈ کرنے میں کچھ وقت لگ سکتا ہے!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


ہم الفاظ سے ویکٹر ایمبیڈنگز بھی حساب کر سکتے ہیں، تاکہ اسے درجہ بندی ماڈل کی تربیت میں استعمال کیا جا سکے (وضاحت کے لیے ویکٹر کے صرف پہلے 20 اجزاء دکھائے گئے ہیں):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

معنوی امبیڈنگز کے بارے میں اچھی بات یہ ہے کہ آپ ویکٹر انکوڈنگ کو سمینٹکس کو بدلنے کے لیے تبدیل کر سکتے ہیں۔ مثال کے طور پر، ہم یہ تلاش کر سکتے ہیں کہ ایسا لفظ کون سا ہے جس کی ویکٹر نمائندگی لفظ *king* اور *woman* کے جتنا ممکن ہو قریب ہو، اور لفظ *man* سے جتنا ممکن ہو دور ہو:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

دونوں CBoW اور Skip-Grams "پیش گوئی کرنے والے" embeddings ہیں، کیونکہ وہ صرف مقامی سیاق و سباق کو مدنظر رکھتے ہیں۔ Word2Vec عالمی سیاق و سباق کا فائدہ نہیں اٹھاتا۔

**FastText**، Word2Vec پر بناتا ہے تاکہ ہر لفظ اور ہر لفظ میں موجود کریکٹر n-گرامز کے لیے ویکٹر کی نمائندگی سیکھ سکے۔ نمائندگی کی قدریں پھر ہر تربیتی مرحلے پر ایک ویکٹر میں اوسط کی جاتی ہیں۔ اگرچہ یہ پری-ٹریننگ میں بہت زیادہ اضافی حساب کتاب شامل کرتا ہے، یہ لفظ کی embeddings کو ذیلی لفظ کی معلومات انکوڈ کرنے کے قابل بناتا ہے۔

ایک اور طریقہ، **GloVe**، کو-اکرنس میٹرکس کے خیال کا فائدہ اٹھاتا ہے، اور نیورل طریقوں کا استعمال کرتے ہوئے کو-اکرنس میٹرکس کو مزید بامعنی اور غیر خطی لفظ ویکٹرز میں تقسیم کرتا ہے۔

آپ مثال کے ساتھ FastText اور GloVe میں embeddings تبدیل کرکے کھیل سکتے ہیں، کیونکہ gensim کئی مختلف لفظ embeddings ماڈلز کی حمایت کرتا ہے۔


## پائی ٹارچ میں پری ٹرینڈ ایمبیڈنگز کا استعمال  

ہم اوپر دی گئی مثال کو اس طرح تبدیل کر سکتے ہیں کہ ہمارے ایمبیڈنگ لیئر میں میٹرکس کو سیمانٹک ایمبیڈنگز، جیسے کہ Word2Vec، سے پہلے سے بھر دیا جائے۔ ہمیں یہ بات ذہن میں رکھنی ہوگی کہ پری ٹرینڈ ایمبیڈنگ کی لغت اور ہمارے ٹیکسٹ کارپس کی لغت ممکن ہے میل نہ کھائیں، اس لیے ہم گمشدہ الفاظ کے لیے وزنوں کو تصادفی اقدار کے ساتھ شروع کریں گے:  


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


اب آئیے اپنے ماڈل کو تربیت دیں۔ نوٹ کریں کہ ماڈل کی تربیت کے لیے لگنے والا وقت پچھلے مثال کی نسبت خاصا زیادہ ہے، کیونکہ ایمبیڈنگ لیئر کا سائز بڑا ہے، اور اس لیے پیرامیٹرز کی تعداد بھی زیادہ ہے۔ نیز، اس وجہ سے، اگر ہم اوورفٹنگ سے بچنا چاہتے ہیں تو ہمیں اپنی ماڈل کو زیادہ مثالوں پر تربیت دینا پڑ سکتی ہے۔


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

ہمارے معاملے میں ہم درستگی میں زبردست اضافہ نہیں دیکھتے، جو ممکنہ طور پر مختلف لغات کی وجہ سے ہے۔ 
مختلف لغات کے مسئلے پر قابو پانے کے لیے، ہم درج ذیل حل میں سے کوئی ایک استعمال کر سکتے ہیں:
* word2vec ماڈل کو ہماری لغت پر دوبارہ تربیت دیں
* ہمارے ڈیٹاسیٹ کو pre-trained word2vec ماڈل کی لغت کے ساتھ لوڈ کریں۔ ڈیٹاسیٹ کو لوڈ کرتے وقت استعمال ہونے والی لغت کا تعین کیا جا سکتا ہے۔

دوسرا طریقہ آسان لگتا ہے، خاص طور پر کیونکہ PyTorch `torchtext` فریم ورک میں ایمبیڈنگز کی بلٹ ان حمایت شامل ہے۔ ہم، مثال کے طور پر، GloVe پر مبنی لغت کو درج ذیل انداز میں قائم کر سکتے ہیں:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

ان ایمبیڈنگز کو استعمال کرتے ہوئے کلاسیفائر کو تربیت دینے کے لیے، ہمیں سب سے پہلے اپنے ڈیٹا سیٹ کو GloVe وکابیلیری کا استعمال کرکے اینکوڈ کرنا ہوگا:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

جیسا کہ ہم نے اوپر دیکھا، تمام ویکٹر ایمبیڈنگز `vocab.vectors` میٹرکس میں محفوظ کی جاتی ہیں۔ اس سے ایمبیڈنگ پرت کے وزن میں ان وزنوں کو آسانی سے نقل کرکے لوڈ کرنا بہت آسان ہو جاتا ہے:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

اب آئیے اپنے ماڈل کو تربیت دیتے ہیں اور دیکھتے ہیں کہ کیا ہمیں بہتر نتائج ملتے ہیں:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

درستگی میں نمایاں اضافے نہ دیکھنے کی ایک وجہ یہ ہے کہ ہمارے ڈیٹا سیٹ کے کچھ الفاظ پری ٹرینڈ GloVe وکبیولری میں موجود نہیں ہیں، اور اس لیے انہیں عملی طور پر نظر انداز کر دیا جاتا ہے۔ اس مسئلے سے نمٹنے کے لیے، ہم اپنے ڈیٹا سیٹ پر اپنے ایمبیڈنگز تربیت دے سکتے ہیں۔


## سیاق و سباق کی ایمبیڈنگز

روایتی پری ٹرینڈ ایمبیڈنگ نمائندگیوں جیسے Word2Vec کی ایک اہم حد الفاظ کے معنی کی تمیز کا مسئلہ ہے۔ اگرچہ پری ٹرینڈ ایمبیڈنگز سیاق و سباق میں الفاظ کے کچھ معانی کو پکڑ سکتی ہیں، ایک ہی ایمبیڈنگ میں ایک لفظ کے تمام ممکنہ معانی کو شامل کر لیا جاتا ہے۔ یہ نیچے کے ماڈلز میں مسائل پیدا کر سکتا ہے، کیونکہ بہت سے الفاظ جیسے لفظ 'play' کے مختلف معانی ہوتے ہیں جو اس سیاق و سباق پر منحصر ہوتے ہیں جن میں وہ استعمال ہوتے ہیں۔

مثال کے طور پر لفظ 'play' ان دو مختلف جملوں میں بالکل مختلف معنی رکھتا ہے:
- میں نے تھیٹر میں ایک **play** دیکھی۔
- جان اپنے دوستوں کے ساتھ **کھیلنا** چاہتا ہے۔

مذکورہ بالا پری ٹرینڈ ایمبیڈنگز لفظ 'play' کے دونوں معانی کو ایک ہی ایمبیڈنگ میں ظاہر کرتی ہیں۔ اس حد کو دور کرنے کے لیے، ہمیں **لینگویج ماڈل** کی بنیاد پر ایمبیڈنگز بنانی ہوں گی، جو متن کے ایک بڑے مجموعہ پر تربیت یافتہ ہوتا ہے اور *جانتا ہے* کہ الفاظ مختلف سیاق و سباق میں کس طرح جڑے جا سکتے ہیں۔ سیاق و سباق کی ایمبیڈنگز پر تبادلہ خیال اس ٹیوٹوریل کے دائرہ کار سے باہر ہے، لیکن ہم اگلے یونٹ میں زبان کے ماڈلز کے بارے میں بات کرتے ہوئے ان پر واپس آئیں گے۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
